# Coding practice: beam search

Greedy decoding keeps the single most likely next token at each step and never goes back. Beam search keeps the K
partial answers with the highest total log-probability instead. Write the step that extends and prunes the beam,
check it on the lecture's example, then run it on two answers from Lab 2's model.

> Save your own copy first: File → Save a copy in Drive.

<details><summary>Hint: scoring and pruning</summary>

An answer's score is the sum of its tokens' log-probabilities, so extending it adds the new token's
log-probability. `sorted(candidates, key=lambda c: c[1], reverse=True)` puts the highest score first.

</details>

In [ ]:
END = "<end>"
MAX_LEN = 10

# A table maps a partial answer, a tuple of tokens, to its most likely next tokens and their log-probabilities.
# The lecture's example, picked up after "Je ne suis pas", with an end token added:
LECTURE = {
    (): [("un", -0.3), ("une", -0.2)],
    ("un",): [("chat", -0.4), ("chien", -0.5)],
    ("une",): [("chat", -0.8), ("chien", -1.2)],
    **{(a, b): [(END, -0.1)] for a in ("un", "une") for b in ("chat", "chien")},
}

# Two math problems answered by Lab 2's model, asked for the number only, as Lab 2's control asks.
# Each table holds the entries a search with K = 1 or 2 reaches.
VOTES = {
    "question": (
        "Two candidates are running for class representative at Sarai's school. If the winner got 3/4 of the "
        "votes and the total number of students who voted in the class was 80, calculate the number of votes "
        "the loser got."
    ),
    "answer": 20,
    "table": {
        (): [("1", -1.47), ("2", -1.65), ("4", -2.28)],
        ("1",): [("5", -0.93), ("2", -1.44), ("0", -1.44)],
        ("2",): [("0", -0.03), ("5", -3.95), ("4", -5.26)],
        ("1", "5"): [(END, -0.01), (".", -4.66), ("0", -7.41)],
        ("2", "0"): [(END, -0.02), (".", -4.23), ("\n", -8.19)],
    },
}

RIBBON = {
    "question": (
        "Monica is wrapping Christmas gifts. She has 6 gifts to wrap for her family, 4 gifts to wrap for her "
        "friends and 2 gifts to wrap for her teachers. She has 144 inches of ribbon and wants to make a bow "
        "for each gift with it. How many inches of ribbon can she use for each gift bow?"
    ),
    "answer": 12,
    "table": {
        (): [("1", -1.14), ("2", -1.3), ("3", -2.19)],
        ("1",): [("2", -1.02), ("4", -1.88), ("8", -1.93)],
        ("2",): [("4", -0.64), (END, -2.27), ("0", -2.43)],
        ("1", "2"): [(END, 0.0), (".", -6.52), ("0", -6.83)],
        ("2", "4"): [(END, 0.0), (".", -7.59), (" inches", -8.96)],
    },
}

In [ ]:
def beam_search(table, K, trace=False):
    """Return the best finished answer, as (tokens, total log-probability), from a beam of width K."""
    beam = [((), 0.0)]                  # partial answers: (tokens so far, total log-probability)
    completed = {}                      # finished answers, kept even if they later drop out of the beam
    for step in range(1, MAX_LEN + 1):
        candidates = []
        for tokens, score in beam:
            if tokens and tokens[-1] == END:
                candidates.append((tokens, score))          # a finished answer stays as it is
                continue
            for token, logprob in table[tokens]:
                # TODO: add the extended answer, tokens + (token,), to candidates with its total log-probability.
                raise NotImplementedError("Extend the answer.")
        # TODO: set beam to the K candidates with the highest total log-probability, best first.
        raise NotImplementedError("Keep the K best.")
        completed.update({t: s for t, s in beam if t[-1] == END})
        if trace:
            print(f"  step {step}:  " + "    ".join(f"{'|'.join(t)} {s:.2f}" for t, s in beam))
        if beam[0][0][-1] == END:       # stop once the best answer is finished
            break
    if completed:
        return max(completed.items(), key=lambda item: item[1])
    return beam[0]                      # nothing finished within MAX_LEN tokens

In [ ]:
tokens, score = beam_search(LECTURE, 1)
assert tokens == ("une", "chat", END), f"With K=1 the search should commit to 'une', as greedy decoding does: {tokens}"
tokens, score = beam_search(LECTURE, 2, trace=True)
assert tokens == ("un", "chat", END) and abs(score + 0.8) < 1e-9, f"With K=2 it should recover 'un chat': {tokens}, {score}"
print("beam_search works.")

Before running the next cell, read each table. In both problems greedy decoding keeps "1" at step 1. Predict, for
each problem, whether a beam of width 2 changes the answer, and whether the change is right.

In [ ]:
for problem in (VOTES, RIBBON):
    print(problem["question"])
    for K in (1, 2):
        print(f"K={K}")
        tokens, score = beam_search(problem["table"], K, trace=True)
        print(f"  answer {''.join(t for t in tokens if t != END)}, total log-probability {score:.2f}")
    print("Correct answer:", problem["answer"], "\n")

## Interpret the outputs

- In the votes problem, greedy decoding kept "1" over "2" at step 1. How much log-probability did each lose over the
  steps after that? Why can "1" lead at step 1 when the answer the model rates highest starts with "2"?
- `beam_search` stops as soon as the best answer in the beam is finished, even if others in the beam are not. Why
  can none of those overtake it?
- In the ribbon problem, K=2 returns an answer the model rates as more likely than greedy's, and it is wrong. What
  is beam search looking for, and why is that not the same as the right answer?

Lab 2's control answers every problem the way K=1 does here.